In [11]:
#  document splitting using RecursiveCharacterTextSplitter in langchain_text_splitters for creating chunks of documents
import os
import re
import shutil
from langchain_core.documents import Document
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

FAISS_PATH = r"C:\SmartTravelData\faiss_hazard_index"
os.makedirs(r"C:\SmartTravelData", exist_ok=True)
shutil.rmtree(FAISS_PATH, ignore_errors=True)

def load_and_chunk_by_paragraph(directory):
    chunks = []
    for filename in os.listdir(directory):
        if not filename.endswith(".txt"):
            continue
        zone_name = filename.replace(".txt", "").replace("_", " ").title()
        filepath = os.path.join(directory, filename)
        with open(filepath, "r", encoding="utf-8") as f:
            content = f.read()
        body = content.split("=" * 60, 1)[-1].strip()
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", body) if p.strip()]
        for para in paragraphs:
            chunks.append(Document(page_content=para, metadata={"zone": zone_name}))
    return chunks

HAZARD_DIR = r"C:\Users\nilad\OneDrive\Desktop\المستندات\Smart Travel\hazard"
chunks = load_and_chunk_by_paragraph(HAZARD_DIR)
chunks = [c for c in chunks if "HAZARD BRIEFING" not in c.page_content and len(c.page_content.strip()) > 20]
print(f"Total clean chunks: {len(chunks)}")

Total clean chunks: 98


In [12]:
# created and embedded the FAISS index from the chunks of documents using HuggingFaceEmbeddings
import os

FAISS_PATH = r"C:\SmartTravelData\faiss_hazard_index"
os.makedirs(FAISS_PATH, exist_ok=True)

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embeddings)
vectorstore.save_local(FAISS_PATH)
print("Saved to:", FAISS_PATH)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1263.82it/s]


Saved to: C:\SmartTravelData\faiss_hazard_index


In [13]:
results = vectorstore.similarity_search("Is Jharkhali safe in monsoon?", k=50)
jharkhali_only = [r for r in results if r.metadata["zone"] == "Jharkhali"][:3]

for r in jharkhali_only:
    print(r.page_content[:80], "->", r.metadata["zone"])

Mosquitoes & Insects: Moderate risk, rising with monsoon humidity. Standard mons -> Jharkhali
Monsoon Weather: Moderate risk of sudden squalls, heavy rain, and rough river co -> Jharkhali
Snakes: Low-Moderate risk, particularly during monsoon (June-September) when flo -> Jharkhali


In [14]:
# Retrieve hazards for a specific zone based on a query
def retrieve_hazards(zone, query, k_search=50, k_return=3):
    results = vectorstore.similarity_search(query, k=k_search)
    zone_matches = [r for r in results if r.metadata["zone"] == zone][:k_return]
    return zone_matches

# Test it
hazards = retrieve_hazards("Dobanki", "Weather and conditions in winter?")
for h in hazards:
    print(h.page_content)
    print()

Monsoon Weather: High risk of sudden squalls, heavy rain, and rough river conditions during June-September. Storms can turn calm conditions violent within 30-40 minutes. Farthest core stop from Gosaba — longest exposure to open water during storms. Boat operations may be suspended with little notice during heavy weather.

Mosquitoes & Insects: Moderate-High risk, rising with monsoon humidity. Dense canopy cover increases insect exposure along the walkway. Sundarban is a malaria-prone zone — carry repellent and consult a doctor about prophylaxis before traveling, especially for multi-day trips.

Royal Bengal Tiger: Moderate-High risk of tiger straying/encounter in this forest block, based on peer-reviewed casualty studies. Risk is highest during pre-monsoon (April-June) and early winter (January-March). This forest block shows elevated straying risk per casualty studies, peaking pre-monsoon and early winter. For boat-based tourists staying on marked watchtower routes, actual risk is low

In [15]:
from langchain_groq import ChatGroq
print("langchain-groq installed successfully")

langchain-groq installed successfully


In [16]:
from dotenv import load_dotenv
import os
import requests

load_dotenv(r"C:\Users\nilad\OneDrive\Desktop\المستندات\Smart Travel\Backend\.env")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")

def call_llm(prompt):
    response = requests.post(
        "https://api.groq.com/openai/v1/chat/completions",
        headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
        json={
            "model": "openai/gpt-oss-120b",
            "messages": [{"role": "user", "content": prompt}],
            "temperature": 0.2
        }
    )
    response.raise_for_status()
    return response.json()["choices"][0]["message"]["content"]



In [17]:
def rewrite_with_context(query, history):
    history_text = "\n".join([f"{h['role']}: {h['content']}" for h in history[-4:]])
    prompt = f"""Given this conversation history, rewrite the latest 
user message into a complete, standalone question that includes all 
necessary context. Output ONLY the rewritten question.

History:
{history_text}

Latest message: {query}
"""
    return call_llm(prompt)

In [18]:
def get_hazard_briefing(zone, query, history=None):
    history = history or []

    standalone_query = rewrite_with_context(query, history) if history else query

    retrieved_chunks = retrieve_hazards(zone, standalone_query)
    context = "\n\n".join([c.page_content for c in retrieved_chunks])

    history_text = "\n".join([f"{h['role']}: {h['content']}" for h in history[-4:]])

    prompt = f"""You are a trip-safety assistant for Sundarban tourists.
Zone: {zone}

Recent conversation:
{history_text}

Answer ONLY using the facts below, which are specific to {zone}.
Do not invent any risk, statistic, or recommendation not explicitly
stated in the context.

If the context does not cover what the user asked, answer naturally
and briefly using whatever related information IS available, without
using phrases like "the information provided" or "the context does
not include." Just speak plainly, e.g. "I don't have winter-specific
weather details for {zone}, but..."

CONTEXT (specific to {zone}):
{context}

QUESTION: {standalone_query}

Answer in 2-4 sentences, warm and practical, naming {zone} explicitly.
"""
    answer = call_llm(prompt)

    history.append({"role": "user", "content": query})
    history.append({"role": "assistant", "content": answer})

    return answer, history

In [19]:
import inspect
print(inspect.getsource(get_hazard_briefing))

def get_hazard_briefing(zone, query, history=None):
    history = history or []

    standalone_query = rewrite_with_context(query, history) if history else query

    retrieved_chunks = retrieve_hazards(zone, standalone_query)
    context = "\n\n".join([c.page_content for c in retrieved_chunks])

    history_text = "\n".join([f"{h['role']}: {h['content']}" for h in history[-4:]])

    prompt = f"""You are a trip-safety assistant for Sundarban tourists.
Zone: {zone}

Recent conversation:
{history_text}

Answer ONLY using the facts below, which are specific to {zone}.
Do not invent any risk, statistic, or recommendation not explicitly
stated in the context.

If the context does not cover what the user asked, answer naturally
and briefly using whatever related information IS available, without
using phrases like "the information provided" or "the context does
not include." Just speak plainly, e.g. "I don't have winter-specific
weather details for {zone}, but..."

CONTEXT (specific to {zo

In [20]:
# Evaluation set — run each question, capture both retrieval and generation output

EVAL_QUESTIONS = [
    {"id": 1,  "zone": "Jharkhali",     "query": "Is it safe in monsoon?"},
    {"id": 2,  "zone": "Dobanki",       "query": "What are the storm and rain risks here?"},
    {"id": 3,  "zone": "Sudhanyakhali", "query": "Any wildlife I should worry about?"},
    {"id": 4,  "zone": "Dobanki",       "query": "What about winter weather?"},
    {"id": 5,  "zone": "Jharkhali",     "query": "What's the best restaurant nearby?"},
    {"id": 6,  "zone": "Dobanki",       "query": "Can we visit with a toddler and elderly parent?"},
    {"id": 7,  "zone": "Netidhopani",   "query": "Is this trip risky for someone with mobility issues?"},
    {"id": 8,  "zone": "Sajnekhali",    "query": "Are there any 5-star hotels?"},
    {"id": 9,  "zone": "Dobanki",       "query": "Is the tiger population increasing?"},
    {"id": 10, "zone": "Dobanki",       "query": "What should I pack?"},
]

results_log = []

for item in EVAL_QUESTIONS:
    print(f"\n{'='*70}")
    print(f"Q{item['id']} | Zone: {item['zone']} | Query: {item['query']}")
    print('='*70)

    # Show what was actually retrieved (for scoring "Retrieval Accuracy")
    retrieved = retrieve_hazards(item["zone"], item["query"])
    print("\n--- RETRIEVED CHUNKS ---")
    for r in retrieved:
        print(f"[{r.metadata['zone']}] {r.page_content[:100]}...")

    # Get the final generated answer (for scoring "Groundedness", "Gap Honesty", "Zone Specificity")
    answer, _ = get_hazard_briefing(item["zone"], item["query"])
    print("\n--- FINAL ANSWER ---")
    print(answer)

    results_log.append({
        "id": item["id"],
        "zone": item["zone"],
        "query": item["query"],
        "retrieved_zones": [r.metadata["zone"] for r in retrieved],
        "answer": answer,
    })

for r in results_log:
    print(f"\n{'='*70}")
    print(f"Q{r['id']} | {r['zone']} | {r['query']}")
    print(f"Retrieved from zones: {r['retrieved_zones']}")
    print(f"Answer: {r['answer']}")


Q1 | Zone: Jharkhali | Query: Is it safe in monsoon?

--- RETRIEVED CHUNKS ---
[Jharkhali] Monsoon Weather: Moderate risk of sudden squalls, heavy rain, and rough river conditions during June...
[Jharkhali] Mosquitoes & Insects: Moderate risk, rising with monsoon humidity. Standard monsoon mosquito exposur...
[Jharkhali] Snakes: Low-Moderate risk, particularly during monsoon (June-September) when flooding pushes snakes ...

--- FINAL ANSWER ---
In Jharkhali the monsoon (June‑September) brings a moderate risk of sudden squalls, heavy rain and rough river conditions, and boat services can be halted with little notice, so keep an eye on weather updates and be ready to adjust plans. Mosquitoes are abundant and the area is malaria‑prone, so use repellent, wear long sleeves and consider prophylaxis before a multi‑day stay. Snakes may appear on the raised paths during floods, so wear closed shoes and ankle‑covering clothing whenever you’re off the boat.

Q2 | Zone: Dobanki | Query: What are 